# V3 Gold runner: verified Silver V3 to new Gold V3 splits

This runner reads one immutable Silver V3 quality report, verifies its source and publication lineage, and optionally creates a new train/validation/test split. It never writes to Silver V1/V2/V3 or Gold V1/V2.

Review the selected snapshot and output paths before enabling `PUBLISH_GOLD_V3`. Test data are checksum-verified as part of the manifest but are not loaded by the modeling runner.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = os.environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_v3_project")
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("Colab checkout has uncommitted changes; resolve them first")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError("Project directory contains non-Git files")
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=PROJECT_DIR, check=True)
SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing project Python sources: {SRC_DIR}")
sys.path[:] = [part for part in sys.path if part != str(SRC_DIR)]
sys.path.insert(0, str(SRC_DIR))
prefix = "vietnamese_topic_classification"
for name in sorted([name for name in sys.modules if name == prefix or name.startswith(prefix + ".")], key=len, reverse=True):
    del sys.modules[name]
importlib.invalidate_caches()

from vietnamese_topic_classification.analysis.news_v3 import build_keyword_report, build_prompt, sample_news_records
from vietnamese_topic_classification.config import get_v3_domain_labels
from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake
from vietnamese_topic_classification.data.gold import GoldConfig, GoldReader, GoldV3Pipeline
from vietnamese_topic_classification.data.silver import SilverReader, SilverV3Pipeline

print("Source revision:", subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
print("Mounted Data Lake:", DATA_LAKE_ROOT)

In [ ]:
SILVER_V3_REPORT_NAME = None  # Set an exact report filename if more than one exists.
reports_dir = DATA_LAKE_ROOT / "04_reports" / "silver" / "v3"
available_reports = sorted(path.name for path in reports_dir.glob("silver_quality_v3_*.json"))
print("Silver V3 reports:", available_reports)
if SILVER_V3_REPORT_NAME is None:
    if len(available_reports) != 1:
        raise ValueError(f"Choose one exact SILVER_V3_REPORT_NAME; found {len(available_reports)}")
    SILVER_V3_REPORT_NAME = available_reports[0]
if Path(SILVER_V3_REPORT_NAME).name != SILVER_V3_REPORT_NAME:
    raise ValueError("Silver V3 report must be a filename, not a path")

silver_v3 = SilverReader(DATA_LAKE_ROOT, SILVER_V3_REPORT_NAME)
verified_silver_v3 = SilverV3Pipeline.verify_reader(silver_v3)
gold_pipeline = GoldV3Pipeline(silver_v3, config=GoldConfig(seed=42))
expected_gold = gold_pipeline.planned_paths(verified_silver_v3["sha256"])
existing_gold = [path for path in expected_gold.values() if path.exists()]
stale_gold = [path for path in expected_gold.values() if path.with_name(path.name + ".uploading").exists()]
print("Verified Silver V3:", verified_silver_v3)
print("Planned Gold V3 outputs:", {name: str(path) for name, path in expected_gold.items()})
if stale_gold or (existing_gold and len(existing_gold) != len(expected_gold)):
    raise FileExistsError("Partial Gold V3 publication or stale upload; inspect Drive before retrying")

gold_v3 = None
PUBLISH_GOLD_V3 = False  # Enable only after reviewing the verified Silver V3 and planned paths.
if existing_gold:
    gold_v3 = GoldReader(DATA_LAKE_ROOT, expected_gold["manifest"].name)
    print("Existing Gold V3 verified:", gold_v3.verify())
elif PUBLISH_GOLD_V3:
    result_gold = gold_pipeline.run()
    gold_v3 = GoldReader(DATA_LAKE_ROOT, expected_gold["manifest"].name)
    print("Published Gold V3:", result_gold["paths"])
    print("Verified Gold V3:", gold_v3.verify())
else:
    print("No Gold V3 snapshot written. Set PUBLISH_GOLD_V3=True after inspection.")


## Gold V3 outputs

The runner writes a versioned manifest and immutable train/validation/test JSONL under `03_gold/v3/`, with its quality report under `04_reports/gold/v3/`. Gold V3 is split anew from all accepted Silver V3 records using the shared leakage-aware splitter; it does not inherit Gold V2 split membership.
